# Group 1 Preprocessing — GSE114007

Scope decided after investigation: **GSE114007 only** for now.

This notebook: download → process → QC. Run top to bottom.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/GaitGenAI_Group1'
for sub in ['data/raw/GSE114007', 'data/clean', 'data/qc']:
    os.makedirs(f'{BASE_DIR}/{sub}', exist_ok=True)
print('Working directory:', BASE_DIR)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/GaitGenAI_Group1


In [2]:
!pip install -q pandas numpy scipy scikit-learn matplotlib openpyxl requests
print('Packages ready.')


Packages ready.


## Step 1 — Download GSE114007

Two download strategies tried in order: GEO's bundle-download endpoint
first, falling back automatically to NCBI's FTP-mirror directory listing
if that fails (known to be flaky for datacenter IPs like Colab's).

In [3]:
import subprocess, tarfile, os, re
import requests

def geo_series_ftp_dir(acc):
    """GSE114007 -> GSE114nnn (NCBI's bucket-directory naming on their FTP mirror)."""
    prefix = acc[:3]
    num = acc[3:]
    bucket = (num[:-3] + 'nnn') if len(num) > 3 else 'nnn'
    return f'{prefix}{bucket}'

def download_via_cgi_endpoint(acc, outdir):
    tarball = f'{outdir}/{acc}_supplementary.tar'
    url = f'https://www.ncbi.nlm.nih.gov/geo/download/?acc={acc}&format=file'
    result = subprocess.run(['curl', '-fsL', '--retry', '2', '--retry-delay', '3',
                              '-o', tarball, url], capture_output=True, text=True)
    if result.returncode != 0 or not tarfile.is_tarfile(tarball):
        if os.path.exists(tarball):
            os.remove(tarball)
        return False
    with tarfile.open(tarball) as t:
        t.extractall(outdir)
    os.remove(tarball)
    return True

def download_via_ftp_mirror(acc, outdir):
    bucket = geo_series_ftp_dir(acc)
    dir_url = f'https://ftp.ncbi.nlm.nih.gov/geo/series/{bucket}/{acc}/suppl/'
    print(f'  Falling back to FTP-mirror directory listing: {dir_url}')
    resp = requests.get(dir_url, timeout=30)
    if resp.status_code != 200:
        print(f'  FTP-mirror listing also failed (HTTP {resp.status_code}).')
        return False
    links = re.findall(r'href="([^"]+)"', resp.text)
    data_links = [
        l for l in links
        if not l.startswith('http://') and not l.startswith('https://')
        and not l.startswith('?') and not l.startswith('..')
        and not l.endswith('/') and l.lower() != 'filelist.txt'
    ]
    if not data_links:
        print(f'  No downloadable files found listed at {dir_url}.')
        return False
    for link in data_links:
        file_url = dir_url + link
        out_path = f'{outdir}/{link}'
        r = requests.get(file_url, timeout=120)
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            f.write(r.content)
        print(f'    downloaded {link}')
    return True

def download_geo_series(acc, base_dir):
    outdir = f'{base_dir}/data/raw/{acc}'
    os.makedirs(outdir, exist_ok=True)
    print(f'Downloading {acc} ...')

    ok = download_via_cgi_endpoint(acc, outdir)
    if not ok:
        ok = download_via_ftp_mirror(acc, outdir)

    if not ok:
        raise RuntimeError(
            f'Both download methods failed for {acc}. Open this page in a '
            f'browser and download supplementary files manually from the '
            f'"Download family" section: '
            f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={acc}\n'
            f'Then upload them into {outdir} via the Colab file browser '
            f'(folder icon on the left) before re-running the processing cell.'
        )

    files = os.listdir(outdir)
    print(f'{acc}: {len(files)} file(s) in {outdir}:')
    for f in sorted(files):
        print('  -', f)
    return files

files_114007 = download_geo_series('GSE114007', BASE_DIR)


  Falling back to FTP-mirror directory listing: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE114nnn/GSE114007/suppl/
    downloaded GSE114007_OA_normalized.counts.txt.gz
    downloaded GSE114007_normal_normalized.counts.txt.gz
    downloaded GSE114007_raw_counts.xlsx
GSE114007: 5 file(s) in /content/drive/MyDrive/GaitGenAI_Group1/data/raw/GSE114007:
  - GSE114007-GPL11154_series_matrix.txt.gz
  - GSE114007-GPL18573_series_matrix.txt.gz
  - GSE114007_OA_normalized.counts.txt.gz
  - GSE114007_normal_normalized.counts.txt.gz
  - GSE114007_raw_counts.xlsx


## Step 2 — Process GSE114007

Loads the two split normal/OA files directly (not `raw_counts.xlsx`,
which is a partial normal-only subset), drops GEO's embedded summary
columns (`Average`, `Max`), derives group from sample names
(`Normal_Cart_*` / `OA_Cart_*`), and parses with a robust reader that
pre-filters malformed lines itself rather than relying on pandas'
lenient-parser fallback (which can silently corrupt a real data column
into all-NaN in some edge cases — verified and avoided here).

In [4]:
import gzip
import io
from collections import Counter
from pathlib import Path
import pandas as pd
import numpy as np
import re as _re

def smart_read_table(path):
    """Robust tab/comma-delimited table reader. Auto-detects delimiter,
    skips leading description lines and trailing footer/notes lines, and
    -- critically -- pre-filters malformed lines itself (by field count)
    BEFORE handing text to pandas, rather than using pandas' on_bad_lines
    lenient mode. That lenient mode was found (via testing) to sometimes
    silently corrupt the last real data column into all-NaN when combined
    with index_col, rather than cleanly skipping just the bad line -- so
    we avoid it entirely and do the line-filtering ourselves."""
    path = Path(path)
    compression = 'gzip' if path.suffix == '.gz' else None
    opener = gzip.open if compression else open

    with opener(path, 'rt', errors='ignore') as fh:
        all_lines = fh.readlines()
    all_lines = [l.rstrip('\n') for l in all_lines]
    if not all_lines:
        raise ValueError(f'{path}: file appears empty')

    sample_lines = all_lines[:20]

    def field_count(line, sep):
        return len(line.split(sep))

    best_sep, best_score, best_modal = None, -1, None
    for sep in ['\t', ',']:
        counts = [field_count(l, sep) for l in sample_lines if l.strip()]
        if not counts:
            continue
        modal_count, modal_freq = Counter(counts).most_common(1)[0]
        if modal_count > 1 and modal_freq > best_score:
            best_sep, best_score, best_modal = sep, modal_freq, modal_count
    if best_sep is None:
        raise ValueError(
            f'{path}: could not detect a consistent delimiter (tried tab, comma) '
            f'from the first {len(sample_lines)} lines. Open the file manually.'
        )

    header_idx = None
    for i, l in enumerate(all_lines):
        if not l.strip():
            continue
        fc = field_count(l, best_sep)
        if fc == best_modal or fc == best_modal - 1:
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f'{path}: could not locate a header line matching the detected field count.')

    header_fc = field_count(all_lines[header_idx], best_sep)

    kept_lines = [all_lines[header_idx]]
    dropped_lines = []
    for l in all_lines[header_idx + 1:]:
        if not l.strip():
            continue
        fc = field_count(l, best_sep)
        if fc == header_fc or fc == best_modal:
            kept_lines.append(l)
        else:
            dropped_lines.append(l)

    if dropped_lines:
        preview = dropped_lines[0][:80] + ('...' if len(dropped_lines[0]) > 80 else '')
        print(f'  [{path.name}] Dropped {len(dropped_lines)} malformed line(s) before '
              f'parsing (field count mismatch) -- first example: {preview!r}')

    buf = io.StringIO('\n'.join(kept_lines))
    df = pd.read_csv(buf, sep=best_sep, index_col=0)

    all_nan_rows = df[df.isna().all(axis=1)]
    if len(all_nan_rows):
        print(f'  [{path.name}] Dropping {len(all_nan_rows)} row(s) that are entirely NaN: '
              f'{list(all_nan_rows.index)}')
        df = df.dropna(how='all')

    all_nan_cols = df.columns[df.isna().all(axis=0)].tolist()
    if all_nan_cols:
        raise ValueError(
            f'{path}: column(s) {all_nan_cols} are entirely NaN after parsing. '
            f'This indicates a real parsing problem, not a clean skip -- '
            f'inspect the raw file directly before trusting this data.'
        )

    return df

def classify_gene_ids(index, sample_size=200):
    idx = pd.Index(index).astype(str)
    sample = idx if len(idx) <= sample_size else idx.to_series().sample(sample_size, random_state=0)
    ensembl_frac = sample.str.match(r'^ENSG\d{11}(\.\d+)?$').mean()
    symbol_like = sample.str.match(r'^[A-Za-z][A-Za-z0-9\-\.]{1,14}$') & ~sample.str.match(r'^ENSG\d{11}(\.\d+)?$')
    symbol_frac = symbol_like.mean()
    if ensembl_frac >= 0.9:
        return 'ensembl', ensembl_frac
    if symbol_frac >= 0.9:
        return 'symbol', symbol_frac
    return 'unknown', max(ensembl_frac, symbol_frac)

def strip_ensembl_version(index):
    return pd.Index(index).astype(str).str.replace(r'\.\d+$', '', regex=True)

def report_missing_and_zero(df, label):
    n_nan = int(df.isna().sum().sum())
    all_zero_cols = df.columns[(df.fillna(0) == 0).all(axis=0)].tolist()
    print(f'[{label}] shape={df.shape}, NaNs={n_nan}, all-zero samples={all_zero_cols or "none"}')

def infer_group_from_colname(col):
    c = col.lower()
    if 'normal' in c:
        return 'normal'
    if 'oa' in c or 'osteoarthrit' in c:
        return 'OA'
    return 'UNKNOWN'

def match_group_file(directory, group_regex):
    directory = Path(directory)
    return [f for f in directory.iterdir()
            if f.is_file() and _re.search(group_regex, f.name, _re.IGNORECASE)
            and f.suffix in ('.gz', '.txt', '.tsv', '.csv')]

SUMMARY_KEYWORDS = ['average', 'mean', 'median', 'max', 'min', 'std', 'sum', 'total']

def drop_summary_columns(df, label):
    bad_cols = [c for c in df.columns if any(k in c.lower() for k in SUMMARY_KEYWORDS)]
    if bad_cols:
        print(f'[{label}] Dropping non-sample summary column(s): {bad_cols}')
        df = df.drop(columns=bad_cols)
    return df

raw_dir = f'{BASE_DIR}/data/raw/GSE114007'

normal_files = match_group_file(raw_dir, r'_normal_normalized')
oa_files = match_group_file(raw_dir, r'_OA_normalized')

if not normal_files or not oa_files:
    raise FileNotFoundError(
        f'Expected a _normal_normalized and an _OA_normalized file in {raw_dir}. '
        f'Found normal: {normal_files}, OA: {oa_files}. List the directory '
        f'yourself with os.listdir("{raw_dir}") and check actual filenames.'
    )

print('Using normal file:', normal_files[0].name)
print('Using OA file:', oa_files[0].name)

normal_df = smart_read_table(normal_files[0])
oa_df = smart_read_table(oa_files[0])

print('normal_df raw shape:', normal_df.shape, '| columns:', list(normal_df.columns))
print('oa_df raw shape:', oa_df.shape, '| columns:', list(oa_df.columns))

normal_df = drop_summary_columns(normal_df, 'normal')
oa_df = drop_summary_columns(oa_df, 'OA')

print('normal_df after cleanup:', normal_df.shape)
print('oa_df after cleanup:', oa_df.shape)

overlap = set(normal_df.columns) & set(oa_df.columns)
if overlap:
    raise ValueError(f'Sample ID overlap between normal and OA files: {overlap}')

gene_common = set(normal_df.index) & set(oa_df.index)
gene_union = set(normal_df.index) | set(oa_df.index)
print(f'Shared genes between the two files: {len(gene_common)}/{len(gene_union)} '
      f'({100*len(gene_common)/len(gene_union):.1f}%)')

counts = pd.concat([normal_df, oa_df], axis=1, join='inner')

numeric_cols = counts.select_dtypes(include='number').columns
dropped = [c for c in counts.columns if c not in numeric_cols]
if dropped:
    print('Dropping non-numeric columns:', dropped)
    counts = counts[numeric_cols]

id_type, frac = classify_gene_ids(counts.index)
print(f'Gene ID format: {id_type} ({frac*100:.0f}% match)')
if id_type == 'ensembl':
    counts.index = strip_ensembl_version(counts.index)

report_missing_and_zero(counts, 'GSE114007')

meta = pd.DataFrame(index=counts.columns)
meta['group'] = [infer_group_from_colname(c) for c in counts.columns]
meta['dataset'] = 'GSE114007'

print()
print('Group counts:', meta['group'].value_counts().to_dict())
print('EXPECTED from the published design: 18 normal / 20 OA (38 total).')
print(f'Got: {counts.shape[1]} total samples.')

n_unknown_group = (meta['group'] == 'UNKNOWN').sum()
if n_unknown_group:
    print(f'WARNING: {n_unknown_group} sample(s) had no Normal/OA match in their '
          f'column name -- inspect manually: '
          f'{meta[meta["group"]=="UNKNOWN"].index.tolist()}')

counts.to_csv(f'{BASE_DIR}/data/clean/GSE114007_counts.csv')
meta.to_csv(f'{BASE_DIR}/data/clean/GSE114007_metadata.csv')
print('Saved to data/clean/GSE114007_counts.csv and GSE114007_metadata.csv')


Using normal file: GSE114007_normal_normalized.counts.txt.gz
Using OA file: GSE114007_OA_normalized.counts.txt.gz
normal_df raw shape: (23710, 20) | columns: ['Normal_Cart_10_8', 'Normal_Cart_2_2', 'Normal_Cart_3_3', 'Normal_Cart_4_4', 'Normal_Cart_5_5', 'Normal_Cart_6_6', 'Normal_Cart_7_3', 'Normal_Cart_9_7', 'normal_01', 'normal_02', 'normal_03', 'normal_04', 'normal_05', 'normal_06', 'normal_07', 'normal_08', 'normal_09', 'normal_10', 'Average Normal', 'Max']
oa_df raw shape: (23710, 22) | columns: ['OA_Cart_1_7', 'OA_Cart_10_9', 'OA_Cart_2_8', 'OA_Cart_3_9', 'OA_Cart_4_10', 'OA_Cart_5_5', 'OA_Cart_6_1', 'OA_Cart_7_2', 'OA_Cart_8_5', 'OA_Cart_9_6', 'OA_01', 'OA_02', 'OA_03', 'OA_04', 'OA_05', 'OA_06', 'OA_07', 'OA_08', 'OA_09', 'OA_10', 'Average OA', 'Max']
[normal] Dropping non-sample summary column(s): ['Average Normal', 'Max']
[OA] Dropping non-sample summary column(s): ['Average OA', 'Max']
normal_df after cleanup: (23710, 18)
oa_df after cleanup: (23710, 20)
Shared genes betwee

## Step 3 — Quick QC: PCA colored by naming convention

Recall the sample names split into two styles within each group
(`Normal_Cart_X_Y` vs `normal_NN`, `OA_Cart_X_Y` vs `OA_NN`) -- likely
the two sequencing platforms (GPL11154 / GPL18573) showing through,
though this was never confirmed via series-matrix metadata. This checks
whether that split lines up with a real expression difference (a batch
effect worth reporting) or not.

In [6]:
print('Min value:', counts.values.min())
print('Max value:', counts.values.max())
print('Any negative values?', (counts.values < 0).any())
print('Fraction of values that are negative:', (counts.values < 0).mean())
print(counts.iloc[:5, :5])

Min value: -4.64758695
Max value: 17.82358868
Any negative values? True
Fraction of values that are negative: 0.39157140003107727
        Normal_Cart_10_8  Normal_Cart_2_2  Normal_Cart_3_3  Normal_Cart_4_4  \
symbol                                                                        
FN1            16.277134        15.429753        15.428266        16.305868   
COMP           15.371944        14.515260        14.813281        14.776144   
MALAT1         15.441039        14.574888        15.053004        14.793931   
CHI3L2          7.645584         5.860772         6.055734         8.496841   
CLU            15.105566        14.493329        14.849689        14.704724   

        Normal_Cart_5_5  
symbol                   
FN1           14.635041  
COMP          14.048698  
MALAT1        14.773987  
CHI3L2         6.743966  
CLU           15.092099  


In [7]:
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

QC_DIR = f'{BASE_DIR}/data/qc'
os.makedirs(QC_DIR, exist_ok=True)

def infer_naming_style(col):
    return 'Cart_style' if '_Cart_' in col else 'short_style'

meta['naming_style'] = [infer_naming_style(c) for c in counts.columns]
print('Naming style counts:', meta['naming_style'].value_counts().to_dict())

# GSE114007's supplementary files, despite being named "*.normalized.counts.*",
# are already log2-transformed normalized expression values (confirmed: value
# range includes negatives, ~-4.6 to 17.8 -- real raw counts are never
# negative). Do NOT apply log2(x+1) again here -- that produces NaN for every
# negative value and silently corrupts the matrix. Use counts as-is.
print(f'Value range: min={counts.values.min():.2f}, max={counts.values.max():.2f}, '
      f'{(counts.values < 0).mean()*100:.1f}% negative -- confirms already log2-scale. '
      f'Using counts directly, no re-transform.')

variances = counts.var(axis=1)
top_genes = variances.sort_values(ascending=False).head(min(2000, len(variances))).index
X = counts.loc[top_genes].T
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(X_scaled)
pc_df = pd.DataFrame(coords, columns=['PC1', 'PC2'], index=X.index).join(meta)

for group_col in ['group', 'naming_style']:
    for pc in ['PC1', 'PC2']:
        groups = [g[pc].values for _, g in pc_df.groupby(group_col)]
        if len(groups) > 1 and all(len(g) > 1 for g in groups):
            f_stat, p_val = stats.f_oneway(*groups)
            ss_between = sum(len(g) * (g.mean() - pc_df[pc].mean())**2 for g in groups)
            ss_total = ((pc_df[pc] - pc_df[pc].mean())**2).sum()
            eta = ss_between / ss_total if ss_total > 0 else float('nan')
            print(f'{pc}: {group_col} explains {eta*100:.1f}% of variance (ANOVA p={p_val:.2e})')
            if eta > 0.25:
                print(f'  -> Real effect (not noise) w.r.t. {group_col}.')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for g, sub in pc_df.groupby('group'):
    axes[0].scatter(sub['PC1'], sub['PC2'], label=g, alpha=0.75, s=40)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].set_title('Colored by group (normal/OA)')
axes[0].legend(fontsize=8)

for s, sub in pc_df.groupby('naming_style'):
    axes[1].scatter(sub['PC1'], sub['PC2'], label=s, alpha=0.75, s=40)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('Colored by naming style (likely platform proxy)')
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(f'{QC_DIR}/GSE114007_pca_check.png', dpi=150)
pc_df.to_csv(f'{QC_DIR}/GSE114007_pca_coordinates.csv')
plt.show()
print(f'Saved plot to {QC_DIR}/GSE114007_pca_check.png')


Naming style counts: {'short_style': 20, 'Cart_style': 18}
Value range: min=-4.65, max=17.82, 39.2% negative -- confirms already log2-scale. Using counts directly, no re-transform.
PC1: group explains 6.5% of variance (ANOVA p=1.23e-01)
PC2: group explains 55.1% of variance (ANOVA p=9.48e-08)
  -> Real effect (not noise) w.r.t. group.
PC1: naming_style explains 57.4% of variance (ANOVA p=3.68e-08)
  -> Real effect (not noise) w.r.t. naming_style.
PC2: naming_style explains 17.7% of variance (ANOVA p=8.52e-03)
Saved plot to /content/drive/MyDrive/GaitGenAI_Group1/data/qc/GSE114007_pca_check.png


## Done — GSE114007 preprocessing complete

**Files saved** on Drive under `GaitGenAI_Group1/`:
- `data/clean/GSE114007_counts.csv` — 38 samples (18 normal / 20 OA), already
  log2-transformed normalized expression (confirmed via negative values in
  the range -4.65 to 17.82 — NOT raw counts, do not log-transform again).
- `data/clean/GSE114007_metadata.csv` — group (normal/OA) + naming_style
  (Cart_style/short_style, a platform proxy) per sample.
- `data/qc/GSE114007_pca_check.png` + `GSE114007_pca_coordinates.csv`.

**Key finding for downstream analysis:** platform (naming_style) is the
dominant source of variance (PC1, 57.4%), exceeding the OA-vs-normal
biological signal (PC2, 55.1%, p=9.5e-08). Both effects are real, and the
design is crossed (not confounded) — both platforms contain both groups.
**Include naming_style/platform as a covariate** in the differential
expression model (`~ naming_style + group` in DESeq2/limma), not just
`~ group` alone.

**Scope:** GSE114007 only. GSE111357 and E-MTAB-7313 deferred (documented
reasons: no usable protein-coding control data; no HPC access).